# Phase 4: Classification ModelsPredicting the attendance band (Low/Medium/High).

In [ ]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier


## 1. Load Data

In [ ]:
train_df = pd.read_csv('../data/processed/train.csv')
val_df = pd.read_csv('../data/processed/val.csv')

leaky_cols = ['Students_Present', 'Attendance_Percentage', 'Total_Enrolled', 'Date']
X_train = train_df.drop(columns=[c for c in leaky_cols + ['Attendance_Class'] if c in train_df.columns])
le = LabelEncoder()
y_train = le.fit_transform(train_df['Attendance_Class'])

X_val = val_df.drop(columns=[c for c in leaky_cols + ['Attendance_Class'] if c in val_df.columns])
y_val = le.transform(val_df['Attendance_Class'])


## 2. Train and Evaluate

In [ ]:
cat_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()
num_cols = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()
preprocessor = ColumnTransformer([('num', StandardScaler(), num_cols), ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols)])

model = XGBClassifier(random_state=42, use_label_encoder=False, eval_metric='mlogloss')
pipeline = Pipeline([('preprocessor', preprocessor), ('classifier', model)])

pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_val)
y_prob = pipeline.predict_proba(X_val)

print(f'Accuracy: {accuracy_score(y_val, y_pred):.4f}')
print(f'F1: {f1_score(y_val, y_pred, average="weighted"):.4f}')
print(f'ROC-AUC: {roc_auc_score(y_val, y_prob, multi_class="ovr"):.4f}')
